# Intelligent Sales Lead Agent — Project 2

A stateful, **conditional** sales workflow built with LangGraph and Gemini.

This notebook builds the entire `02-sales-agent` project from scratch:
installs dependencies, writes every source file to disk, configures Gemini
via Colab Secrets, runs both a qualified and an unqualified lead through
the graph, runs the test suite, prints a validation summary, and creates a
GitHub-ready ZIP backup.

Project 1 in this portfolio demonstrated a **sequential** graph. Project 2
demonstrates **state-driven conditional routing**: the graph itself decides
whether a lead goes to `research → outreach` or to `nurture`, based on
state written by an LLM qualification call earlier in the graph.


In [1]:
!pip install -q langgraph langchain langchain-core langchain-google-genai pydantic python-dotenv pytest grandalf
print("Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.1 MB/s eta 0:00:00
Dependencies installed.


In [2]:
import os

PROJECT_ROOT = "/content/langgraph-enterprise-workflows/02-sales-agent"
DIRS = [
    PROJECT_ROOT,
    f"{PROJECT_ROOT}/tests",
    f"{PROJECT_ROOT}/examples",
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Created:", DIRS)


Project root: /content/langgraph-enterprise-workflows/02-sales-agent
Created: ['/content/langgraph-enterprise-workflows/02-sales-agent', '/content/langgraph-enterprise-workflows/02-sales-agent/tests', '/content/langgraph-enterprise-workflows/02-sales-agent/examples']


In [3]:
%%writefile requirements.txt
langgraph>=0.2,<2
langchain>=0.3
langchain-core>=0.3
langchain-google-genai>=2.0
pydantic>=2.0
python-dotenv>=1.0
pytest>=8.0
grandalf>=0.8  # optional: enables graph.get_graph().draw_ascii() for demos


Writing requirements.txt


In [4]:
%%writefile .gitignore
# Secrets
.env

# Python
__pycache__/
*.py[cod]
*.egg-info/
.pytest_cache/
.venv/
venv/

# OS
.DS_Store

# Notebook checkpoints
.ipynb_checkpoints/

# Packaging artifacts
*.zip


Writing .gitignore


In [5]:
%%writefile .env.example
# Google AI Studio API key. Never commit a real key.
GOOGLE_API_KEY=your-google-api-key-here

# Gemini model to use for all LLM calls.
GEMINI_MODEL=gemini-3.6-flash

# Minimum qualification_score (0-100) required to route a lead to the
# qualified (research -> outreach) path instead of nurture.
QUALIFICATION_THRESHOLD=60

# Max retry attempts for transient LLM/API failures (in addition to the
# first attempt).
MAX_LLM_RETRIES=2

# Python logging level: DEBUG, INFO, WARNING, ERROR.
LOG_LEVEL=INFO


Writing .env.example


In [6]:
%%writefile config.py
"""
config.py

Centralizes environment/configuration handling for the sales lead agent.
No secrets are ever hardcoded here -- everything is read from environment
variables, with sane (non-secret) defaults for local development.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import load_dotenv

# Load a local .env file if present. Safe no-op if it doesn't exist.
load_dotenv()


@dataclass(frozen=True)
class Settings:
    """Runtime configuration for the sales lead agent."""

    google_api_key: str | None
    gemini_model: str
    qualification_threshold: int
    max_llm_retries: int
    log_level: str


def _get_int(name: str, default: int) -> int:
    raw = os.environ.get(name)
    if raw is None or raw.strip() == "":
        return default
    try:
        return int(raw)
    except ValueError:
        return default


def load_settings() -> Settings:
    """Build a :class:`Settings` instance from the current environment.

    Called lazily (not at import time) so that tests and notebooks can set
    environment variables *before* the settings are constructed.
    """

    return Settings(
        google_api_key=os.environ.get("GOOGLE_API_KEY"),
        gemini_model=os.environ.get("GEMINI_MODEL", "gemini-3.6-flash"),
        qualification_threshold=_get_int("QUALIFICATION_THRESHOLD", 60),
        max_llm_retries=_get_int("MAX_LLM_RETRIES", 2),
        log_level=os.environ.get("LOG_LEVEL", "INFO"),
    )


settings = load_settings()


Writing config.py


In [7]:
%%writefile state.py
"""
state.py

Defines the shared graph state for the Intelligent Sales Lead Agent.

This file intentionally contains NO business logic. It only describes the
shape of the data that flows between LangGraph nodes.
"""

from __future__ import annotations

from typing import Any, Literal, TypedDict


class SalesState(TypedDict, total=False):
    """Shared state passed between every node in the sales lead graph.

    Fields are grouped by the stage of the workflow that populates them.
    All fields are optional (``total=False``) because the state is built up
    incrementally as it flows through the graph -- a freshly submitted lead
    will not yet have a ``qualification_score``, for example.
    """

    # --- Identity -----------------------------------------------------
    lead_id: str
    """Stable identifier for the lead. Generated during normalization if
    the caller did not supply one."""

    # --- Raw / normalized lead data ------------------------------------
    lead_name: str
    """The contact's name."""

    company: str
    """The company the lead works for."""

    role: str
    """The lead's job title / role (e.g. 'CTO', 'VP Engineering')."""

    industry: str
    """The industry the lead's company operates in. Optional."""

    company_size: int
    """Approximate employee count. Optional, normalized to an int."""

    need: str
    """Free-text description of the lead's business need/problem."""

    budget: str
    """Free-text budget figure as supplied by the lead (e.g. '$75000').
    Kept as a string because leads rarely give clean numeric input, but
    normalization will strip stray characters where possible."""

    urgency: str
    """Free-text urgency signal (e.g. 'High', 'Low', 'Q3 rollout')."""

    # --- Qualification --------------------------------------------------
    qualification_score: int
    """0-100 score produced by the qualification node."""

    qualification_status: Literal["qualified", "unqualified"]
    """Categorical qualification outcome derived from the score/LLM."""

    qualification_reason: str
    """Human-readable justification for the qualification outcome."""

    qualification_strengths: list[str]
    """Positive signals identified during qualification."""

    qualification_concerns: list[str]
    """Risk factors / concerns identified during qualification."""

    # --- Downstream content ---------------------------------------------
    research: str
    """LLM-generated sales research brief (qualified leads only)."""

    outreach_message: str
    """Personalized outreach message (qualified leads only)."""

    nurture_message: str
    """Relationship-preserving message (unqualified leads only)."""

    # --- Workflow control -------------------------------------------------
    next_action: Literal["sales_outreach", "nurture", "human_review"]
    """What the sales system should do next with this lead."""

    # --- Observability / error handling -----------------------------------
    errors: list[str]
    """Accumulated error messages. Never silently discarded."""

    metadata: dict[str, Any]
    """Free-form bag for timing info, retry counters, node history, etc."""


Writing state.py


In [8]:
%%writefile prompts.py
"""
prompts.py

All prompts used by the sales lead agent live here, and only here, so that
nodes stay small and prompt copy can be reviewed/edited in one place.
"""

from __future__ import annotations

QUALIFICATION_SYSTEM_PROMPT = """\
You are a senior B2B sales qualification analyst. You evaluate inbound \
leads for a company that sells AI workflow automation software.

Assess the lead across these dimensions:
- Business need: is the stated problem significant enough to justify a paid solution?
- Buyer relevance: does this person's role suggest influence over a purchase decision?
- Budget: does the stated budget suggest realistic purchasing potential?
- Company fit: does the company profile (industry, size) fit a mid-to-enterprise \
  B2B software buyer?
- Urgency: is there evidence the problem needs near-term attention?
- Overall fit: combine the above into a single opportunity assessment.

Score the lead from 0 to 100, where 100 is a perfect-fit, ready-to-buy lead and 0 is \
a lead with no realistic fit at all. Be honest and discriminating -- most leads are \
NOT a perfect 90+, and a lead with a vague need, an unclear role, or no budget signal \
should score low. Do not default to a middling score; use the full range.

Respond with structured output only, following the provided schema exactly."""

QUALIFICATION_USER_PROMPT = """\
Evaluate the following lead:

Name: {lead_name}
Company: {company}
Role: {role}
Industry: {industry}
Company size: {company_size}
Stated need: {need}
Budget: {budget}
Urgency: {urgency}

Provide your qualification assessment."""


LEAD_RESEARCH_SYSTEM_PROMPT = """\
You are a B2B sales research analyst. You write brief, useful internal research \
notes that a salesperson would read for two minutes before their first call with \
a lead. You are working ONLY from the information given to you -- you do not have \
access to the internet, the company's website, or any external database. This is \
an LLM-based synthesis, not verified external research, and your output should \
read that way: reasonable inference from the stated facts, not invented specifics \
about the company (no fabricated statistics, news, or quotes).

Cover, briefly:
- Likely business priorities for someone in this role at this kind of company
- Probable pain points related to the stated need
- Relevant AI/automation opportunities
- Likely decision criteria for a purchase like this
- A potential value proposition
- A suggested messaging angle for outreach

Keep the whole brief under 200 words. Use short paragraphs or a light list -- no \
headers, no markdown tables."""

LEAD_RESEARCH_USER_PROMPT = """\
Lead:
Name: {lead_name}
Company: {company}
Role: {role}
Industry: {industry}
Company size: {company_size}
Stated need: {need}
Budget: {budget}
Urgency: {urgency}

Qualification summary: {qualification_reason}

Write the research brief."""


OUTREACH_SYSTEM_PROMPT = """\
You are an experienced B2B sales rep writing a first-touch outreach email. \
The email must be:
- Personalized to the specific lead and their stated need
- Concise (under 150 words)
- Professional, not pushy or hype-driven
- Specific, referencing only facts that were actually provided
- Free of fabricated claims about the company (no invented statistics, incidents, \
  or "we noticed you're losing millions" style claims unless that information was \
  actually given to you)
- Ending with a low-friction call to action (e.g. a short call)

Sign off as "The Team" with no company name invented."""

OUTREACH_USER_PROMPT = """\
Lead:
Name: {lead_name}
Company: {company}
Role: {role}
Stated need: {need}

Qualification reason: {qualification_reason}

Research brief:
{research}

Write the outreach email. Return only the email body."""


NURTURE_SYSTEM_PROMPT = """\
You are a B2B sales rep writing a short nurture note to a lead who is not yet a \
strong fit -- for reasons like unclear budget, uncertain timing, or a need that \
doesn't yet match what we offer. The note must:
- Stay warm and professional, never dismissive
- Avoid aggressive sales language or a hard pitch
- Acknowledge, gently, that now may not be the right time or fit
- Keep the door open for the future
- Suggest a light, low-pressure next step (e.g. staying in touch, revisiting in a \
  few months, sharing resources)

Keep it under 120 words. Sign off as "The Team"."""

NURTURE_USER_PROMPT = """\
Lead:
Name: {lead_name}
Company: {company}
Role: {role}
Stated need: {need}

Qualification reason: {qualification_reason}
Concerns: {concerns}

Write the nurture email. Return only the email body."""


Writing prompts.py


In [9]:
%%writefile validators.py
"""
validators.py

Validates raw lead input before it is allowed to enter the LangGraph
workflow. Malformed input must never reach the qualification LLM call.
"""

from __future__ import annotations

from dataclasses import dataclass, field

REQUIRED_FIELDS = ("lead_name", "company", "role", "need")
OPTIONAL_FIELDS = ("industry", "company_size", "budget", "urgency")


@dataclass
class ValidationResult:
    """Outcome of validating a raw lead dict."""

    is_valid: bool
    errors: list[str] = field(default_factory=list)


def _is_blank(value: object) -> bool:
    return value is None or (isinstance(value, str) and value.strip() == "")


def validate_lead(raw_lead: dict) -> ValidationResult:
    """Validate a raw lead dictionary before normalization.

    This checks presence/shape only -- it does not make any judgment about
    whether the lead is a *good* one. That judgment belongs to the
    qualification node, not here.
    """

    errors: list[str] = []

    if not isinstance(raw_lead, dict):
        return ValidationResult(is_valid=False, errors=["Lead input must be a dictionary."])

    # Required string fields must be present and non-blank.
    for field_name in REQUIRED_FIELDS:
        value = raw_lead.get(field_name)
        if _is_blank(value):
            errors.append(f"Missing required field: '{field_name}'.")
        elif not isinstance(value, str):
            errors.append(f"Field '{field_name}' must be a string.")

    # lead_name minimum plausibility check.
    lead_name = raw_lead.get("lead_name")
    if isinstance(lead_name, str) and 0 < len(lead_name.strip()) < 2:
        errors.append("Field 'lead_name' is too short to be a valid name.")

    # company_size, if present, must be a positive integer (or numeric string).
    if "company_size" in raw_lead and not _is_blank(raw_lead.get("company_size")):
        company_size = raw_lead["company_size"]
        try:
            size_int = int(company_size)
            if size_int <= 0:
                errors.append("Field 'company_size' must be a positive integer.")
        except (TypeError, ValueError):
            errors.append("Field 'company_size' must be a valid integer.")

    # Optional string fields, if present, must actually be strings.
    for field_name in ("industry", "budget", "urgency"):
        if field_name in raw_lead and not _is_blank(raw_lead.get(field_name)):
            if not isinstance(raw_lead[field_name], str):
                errors.append(f"Field '{field_name}' must be a string if provided.")

    return ValidationResult(is_valid=len(errors) == 0, errors=errors)


Writing validators.py


In [10]:
%%writefile nodes.py
"""
nodes.py

The actual LangGraph node functions for the sales lead agent. Each node:

1. Reads only the state it needs.
2. Performs one responsibility.
3. Returns a partial state update (never mutates state in place).
4. Handles failures by recording them in ``errors`` rather than raising.
5. Logs its execution.

The LLM objects are built lazily via small factory functions
(``get_qualification_llm``, ``get_research_llm``, ``get_content_llm``) rather
than at import time. This is what makes the nodes testable: unit tests
monkeypatch these factories to return a deterministic fake model instead of
calling Gemini.
"""

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field, field_validator

import prompts
from config import settings
from utils import TransientLLMError, call_with_retry, generate_lead_id, logger
from validators import validate_lead


# ---------------------------------------------------------------------------
# Structured qualification schema
# ---------------------------------------------------------------------------


class QualificationResult(BaseModel):
    """Structured output produced by the qualification LLM call."""

    score: int = Field(ge=0, le=100, description="Overall lead fit score, 0-100.")
    status: Literal["qualified", "unqualified"] = Field(
        description="Categorical qualification outcome."
    )
    reason: str = Field(min_length=1, description="Justification for the score/status.")
    strengths: list[str] = Field(default_factory=list)
    concerns: list[str] = Field(default_factory=list)

    @field_validator("reason")
    @classmethod
    def _reason_not_blank(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("reason must not be blank")
        return value


# ---------------------------------------------------------------------------
# LLM factories (overridden in tests)
# ---------------------------------------------------------------------------


def _base_chat_model():
    """Build the underlying Gemini chat model.

    Imported lazily so that the ``langchain_google_genai`` dependency is
    only required when a node actually needs to call the model (tests never
    hit this function because they monkeypatch the factories below).
    """

    from langchain_google_genai import ChatGoogleGenerativeAI

    if not settings.google_api_key:
        raise TransientLLMError(
            "GOOGLE_API_KEY is not configured; cannot call the Gemini model."
        )

    return ChatGoogleGenerativeAI(
        model=settings.gemini_model,
        google_api_key=settings.google_api_key,
        temperature=0.3,
    )


def get_qualification_llm():
    """Return a model configured to emit :class:`QualificationResult`."""

    return _base_chat_model().with_structured_output(QualificationResult)


def get_research_llm():
    """Return a plain chat model used for the research brief."""

    return _base_chat_model()


def get_content_llm():
    """Return a plain chat model used for outreach/nurture copy."""

    return _base_chat_model()


# ---------------------------------------------------------------------------
# Node: normalize_lead
# ---------------------------------------------------------------------------


def normalize_lead(state: dict) -> dict:
    """Clean and validate the raw lead before it enters the rest of the graph."""

    logger.info("Lead normalization started")

    validation = validate_lead(state)
    if not validation.is_valid:
        logger.info("Lead normalization failed validation: %s", validation.errors)
        return {
            "errors": list(state.get("errors", [])) + validation.errors,
            "next_action": "human_review",
            "metadata": {**state.get("metadata", {}), "validation_failed": True},
        }

    lead_id = state.get("lead_id") or generate_lead_id()

    def _clean_str(value: object) -> str:
        return value.strip() if isinstance(value, str) else ""

    company_size_raw = state.get("company_size")
    company_size: int | None
    if company_size_raw in (None, ""):
        company_size = None
    else:
        try:
            company_size = int(company_size_raw)
        except (TypeError, ValueError):
            company_size = None

    budget_raw = _clean_str(state.get("budget", "")) or "Not specified"
    urgency = _clean_str(state.get("urgency", "")) or "Not specified"
    industry = _clean_str(state.get("industry", "")) or "Not specified"

    normalized = {
        "lead_id": lead_id,
        "lead_name": _clean_str(state.get("lead_name", "")),
        "company": _clean_str(state.get("company", "")),
        "role": _clean_str(state.get("role", "")),
        "industry": industry,
        "need": _clean_str(state.get("need", "")),
        "budget": budget_raw,
        "urgency": urgency,
        "errors": list(state.get("errors", [])),
        "metadata": {**state.get("metadata", {}), "normalized": True},
    }
    if company_size is not None:
        normalized["company_size"] = company_size

    logger.info("Lead normalization completed | lead_id=%s", lead_id)
    return normalized


# ---------------------------------------------------------------------------
# Node: qualify_lead
# ---------------------------------------------------------------------------


def qualify_lead(state: dict) -> dict:
    """Score and classify the lead using the qualification LLM.

    LangGraph -- not the LLM -- makes the final routing decision. This node
    only produces the structured qualification result and stores it in
    state; ``route_lead`` (in graph.py) reads that state to pick a path.
    """

    lead_id = state.get("lead_id", "unknown")
    logger.info("Qualification started | lead_id=%s", lead_id)

    prompt_kwargs = {
        "lead_name": state.get("lead_name", ""),
        "company": state.get("company", ""),
        "role": state.get("role", ""),
        "industry": state.get("industry", "Not specified"),
        "company_size": state.get("company_size", "Not specified"),
        "need": state.get("need", ""),
        "budget": state.get("budget", "Not specified"),
        "urgency": state.get("urgency", "Not specified"),
    }

    def _invoke() -> QualificationResult:
        llm = get_qualification_llm()
        messages = [
            ("system", prompts.QUALIFICATION_SYSTEM_PROMPT),
            ("user", prompts.QUALIFICATION_USER_PROMPT.format(**prompt_kwargs)),
        ]
        try:
            result = llm.invoke(messages)
        except TransientLLMError:
            raise
        except Exception as exc:  # noqa: BLE001 - genuinely unknown provider errors
            raise TransientLLMError(str(exc)) from exc

        if isinstance(result, QualificationResult):
            return result
        # Fallback: some providers return a dict-like structured output.
        try:
            return QualificationResult.model_validate(result)
        except Exception as exc:  # noqa: BLE001
            raise TransientLLMError(f"Malformed structured output: {exc}") from exc

    try:
        qualification = call_with_retry(_invoke, node_name="qualify_lead")
    except TransientLLMError as exc:
        logger.info("Qualification failed after retries | lead_id=%s | %s", lead_id, exc)
        return {
            "errors": list(state.get("errors", [])) + [f"Qualification node failed: {exc}"],
            "next_action": "human_review",
        }

    # Enforce the configurable threshold as the authoritative routing signal,
    # even if the model's own "status" field disagrees with its own score.
    status: Literal["qualified", "unqualified"] = (
        "qualified" if qualification.score >= settings.qualification_threshold else "unqualified"
    )

    logger.info(
        "Qualification score: %d | status=%s | lead_id=%s",
        qualification.score,
        status,
        lead_id,
    )

    return {
        "qualification_score": qualification.score,
        "qualification_status": status,
        "qualification_reason": qualification.reason,
        "qualification_strengths": qualification.strengths,
        "qualification_concerns": qualification.concerns,
    }


# ---------------------------------------------------------------------------
# Node: research_lead (qualified path)
# ---------------------------------------------------------------------------


def research_lead(state: dict) -> dict:
    """Produce an LLM-based lead research synthesis.

    This is explicitly an LLM-based synthesis from the information already
    on the lead -- not live web research. No external research tool is
    wired up in this project.
    """

    lead_id = state.get("lead_id", "unknown")
    logger.info("Lead research started | lead_id=%s", lead_id)

    prompt_kwargs = {
        "lead_name": state.get("lead_name", ""),
        "company": state.get("company", ""),
        "role": state.get("role", ""),
        "industry": state.get("industry", "Not specified"),
        "company_size": state.get("company_size", "Not specified"),
        "need": state.get("need", ""),
        "budget": state.get("budget", "Not specified"),
        "urgency": state.get("urgency", "Not specified"),
        "qualification_reason": state.get("qualification_reason", ""),
    }

    def _invoke() -> str:
        llm = get_research_llm()
        messages = [
            ("system", prompts.LEAD_RESEARCH_SYSTEM_PROMPT),
            ("user", prompts.LEAD_RESEARCH_USER_PROMPT.format(**prompt_kwargs)),
        ]
        try:
            response = llm.invoke(messages)
        except Exception as exc:  # noqa: BLE001
            raise TransientLLMError(str(exc)) from exc
        content = getattr(response, "content", response)
        if not isinstance(content, str) or not content.strip():
            raise TransientLLMError("Research LLM returned empty content.")
        return content.strip()

    try:
        research_text = call_with_retry(_invoke, node_name="research_lead")
    except TransientLLMError as exc:
        logger.info("Lead research failed after retries | lead_id=%s | %s", lead_id, exc)
        return {
            "errors": list(state.get("errors", [])) + [f"Research node failed: {exc}"],
            "research": "",
        }

    logger.info("Lead research completed | lead_id=%s", lead_id)
    return {"research": f"[LLM-based lead research synthesis]\n{research_text}"}


# ---------------------------------------------------------------------------
# Node: generate_outreach (qualified path)
# ---------------------------------------------------------------------------


def generate_outreach(state: dict) -> dict:
    """Generate a personalized outreach message for a qualified lead."""

    lead_id = state.get("lead_id", "unknown")
    logger.info("Outreach generation started | lead_id=%s", lead_id)

    prompt_kwargs = {
        "lead_name": state.get("lead_name", ""),
        "company": state.get("company", ""),
        "role": state.get("role", ""),
        "need": state.get("need", ""),
        "qualification_reason": state.get("qualification_reason", ""),
        "research": state.get("research", ""),
    }

    def _invoke() -> str:
        llm = get_content_llm()
        messages = [
            ("system", prompts.OUTREACH_SYSTEM_PROMPT),
            ("user", prompts.OUTREACH_USER_PROMPT.format(**prompt_kwargs)),
        ]
        try:
            response = llm.invoke(messages)
        except Exception as exc:  # noqa: BLE001
            raise TransientLLMError(str(exc)) from exc
        content = getattr(response, "content", response)
        if not isinstance(content, str) or not content.strip():
            raise TransientLLMError("Outreach LLM returned empty content.")
        return content.strip()

    try:
        message = call_with_retry(_invoke, node_name="generate_outreach")
    except TransientLLMError as exc:
        logger.info("Outreach generation failed after retries | lead_id=%s | %s", lead_id, exc)
        return {
            "errors": list(state.get("errors", [])) + [f"Outreach node failed: {exc}"],
            "outreach_message": "",
            "next_action": "human_review",
        }

    logger.info("Outreach generation completed | lead_id=%s", lead_id)
    return {"outreach_message": message, "next_action": "sales_outreach"}


# ---------------------------------------------------------------------------
# Node: generate_nurture (unqualified path)
# ---------------------------------------------------------------------------


def generate_nurture(state: dict) -> dict:
    """Generate a relationship-preserving nurture message for a weaker-fit lead."""

    lead_id = state.get("lead_id", "unknown")
    logger.info("Nurture generation started | lead_id=%s", lead_id)

    prompt_kwargs = {
        "lead_name": state.get("lead_name", ""),
        "company": state.get("company", ""),
        "role": state.get("role", ""),
        "need": state.get("need", ""),
        "qualification_reason": state.get("qualification_reason", ""),
        "concerns": "; ".join(state.get("qualification_concerns", [])) or "Not specified",
    }

    def _invoke() -> str:
        llm = get_content_llm()
        messages = [
            ("system", prompts.NURTURE_SYSTEM_PROMPT),
            ("user", prompts.NURTURE_USER_PROMPT.format(**prompt_kwargs)),
        ]
        try:
            response = llm.invoke(messages)
        except Exception as exc:  # noqa: BLE001
            raise TransientLLMError(str(exc)) from exc
        content = getattr(response, "content", response)
        if not isinstance(content, str) or not content.strip():
            raise TransientLLMError("Nurture LLM returned empty content.")
        return content.strip()

    try:
        message = call_with_retry(_invoke, node_name="generate_nurture")
    except TransientLLMError as exc:
        logger.info("Nurture generation failed after retries | lead_id=%s | %s", lead_id, exc)
        return {
            "errors": list(state.get("errors", [])) + [f"Nurture node failed: {exc}"],
            "nurture_message": "",
            "next_action": "human_review",
        }

    logger.info("Nurture generation completed | lead_id=%s", lead_id)
    return {"nurture_message": message, "next_action": "nurture"}


Writing nodes.py


In [11]:
%%writefile graph.py
"""
graph.py

Builds the actual LangGraph workflow for the sales lead agent.

Two conditional edges drive the whole project:

1. After ``normalize`` -- obviously invalid leads never reach the LLM.
2. After ``qualify`` -- qualified leads go to research/outreach, everyone
   else goes to nurture.

Both routing decisions are made with ``add_conditional_edges`` reading
state that LangGraph itself manages. There is no ordinary Python
``if/else`` wrapped around graph execution to fake branching.
"""

from __future__ import annotations

from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import RetryPolicy

from nodes import (
    generate_nurture,
    generate_outreach,
    normalize_lead,
    qualify_lead,
    research_lead,
)
from state import SalesState
from utils import TransientLLMError, logger

# Nodes that call an LLM get a graph-level retry policy on top of the
# node-internal retry helper, so transient provider failures are retried
# even if they escape the node's own retry loop. Validation-style errors
# are plain ValueErrors/dict updates, never exceptions, so they are never
# retried here.
_LLM_RETRY_POLICY = RetryPolicy(max_attempts=2, retry_on=(TransientLLMError,))


def route_after_normalize(state: SalesState) -> str:
    """Return the next route key after normalization.

    An obviously invalid lead (missing required fields) must never reach
    the qualification LLM call -- it is routed straight to the end of the
    graph with its errors already recorded in state.
    """

    if state.get("errors"):
        logger.info("Routing lead: invalid_input")
        return "invalid"
    logger.info("Routing lead: valid_input")
    return "valid"


def route_lead(state: SalesState) -> str:
    """Return the next route key after qualification.

    This is the central conditional-routing decision of the project: the
    graph -- not ad-hoc Python control flow -- decides whether a lead
    proceeds to research/outreach or to nurture, based purely on state
    that the qualification node wrote.
    """

    if state.get("errors") and not state.get("qualification_status"):
        # Qualification itself failed (e.g. exhausted retries).
        logger.info("Routing lead: qualification_failed -> nurture")
        return "unqualified"

    status = state.get("qualification_status", "unqualified")
    logger.info("Routing lead: %s", status)
    return status


def build_graph() -> CompiledStateGraph:
    """Construct and compile the sales lead StateGraph."""

    builder = StateGraph(SalesState)

    builder.add_node("normalize", normalize_lead)
    builder.add_node("qualify", qualify_lead, retry_policy=_LLM_RETRY_POLICY)
    builder.add_node("research", research_lead, retry_policy=_LLM_RETRY_POLICY)
    builder.add_node("outreach", generate_outreach, retry_policy=_LLM_RETRY_POLICY)
    builder.add_node("nurture", generate_nurture, retry_policy=_LLM_RETRY_POLICY)

    builder.add_edge(START, "normalize")

    builder.add_conditional_edges(
        "normalize",
        route_after_normalize,
        {
            "valid": "qualify",
            "invalid": END,
        },
    )

    builder.add_conditional_edges(
        "qualify",
        route_lead,
        {
            "qualified": "research",
            "unqualified": "nurture",
        },
    )

    builder.add_edge("research", "outreach")
    builder.add_edge("outreach", END)
    builder.add_edge("nurture", END)

    return builder.compile()


def get_graph_ascii() -> str:
    """Return a printable representation of the compiled graph, for demos."""

    graph = build_graph()
    try:
        return graph.get_graph().draw_ascii()
    except Exception:  # noqa: BLE001 - ascii rendering needs an extra dep
        return graph.get_graph().print_ascii() or ""


Writing graph.py


In [12]:
%%writefile utils.py
"""
utils.py

Small, dependency-light helpers shared across nodes: logging setup, lead ID
generation, and a retry helper for transient LLM/API failures.
"""

from __future__ import annotations

import logging
import time
import uuid
from typing import Callable, TypeVar

from config import settings

T = TypeVar("T")


def configure_logging() -> logging.Logger:
    """Configure and return the package-wide logger.

    Safe to call multiple times (e.g. once from app.py, once from tests) --
    handlers are only attached once.
    """

    logger = logging.getLogger("sales_agent")
    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter("%(levelname)s | %(message)s")
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(getattr(logging, settings.log_level.upper(), logging.INFO))
        logger.propagate = False
    return logger


logger = configure_logging()


def generate_lead_id() -> str:
    """Generate a short, stable-looking lead identifier."""

    return f"lead_{uuid.uuid4().hex[:10]}"


class TransientLLMError(Exception):
    """Raised by LLM wrapper calls to signal a retryable failure.

    This is distinct from validation errors (which are never retried) and
    from unrecoverable errors (which are logged and surfaced, not retried
    forever).
    """


def call_with_retry(
    fn: Callable[[], T],
    *,
    max_retries: int | None = None,
    base_delay_seconds: float = 0.0,
    node_name: str = "node",
) -> T:
    """Call ``fn`` with retry-on-``TransientLLMError`` semantics.

    Only :class:`TransientLLMError` triggers a retry. Any other exception
    (including validation-style errors) is raised immediately -- retrying
    invalid input is never appropriate.
    """

    attempts = (max_retries if max_retries is not None else settings.max_llm_retries) + 1
    last_error: Exception | None = None

    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except TransientLLMError as exc:
            last_error = exc
            logger.info(
                "%s: transient failure on attempt %d/%d: %s",
                node_name,
                attempt,
                attempts,
                exc,
            )
            if attempt < attempts and base_delay_seconds > 0:
                time.sleep(base_delay_seconds)

    assert last_error is not None
    raise last_error


Writing utils.py


In [13]:
%%writefile app.py
"""
app.py

Interactive CLI for the Intelligent Sales Lead Agent.

Usage:
    python app.py                 # interactive prompts
    python app.py --example qualified    # run examples/qualified_lead.json
    python app.py --example unqualified  # run examples/unqualified_lead.json
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

from graph import build_graph
from utils import logger

BANNER = "=" * 60


def _prompt(label: str, required: bool = True) -> str:
    while True:
        value = input(f"{label}: ").strip()
        if value or not required:
            return value
        print(f"  '{label}' is required.")


def collect_lead_interactively() -> dict:
    print(BANNER)
    print("INTELLIGENT SALES LEAD AGENT")
    print(BANNER)
    print()
    lead = {
        "lead_name": _prompt("Lead name"),
        "company": _prompt("Company"),
        "role": _prompt("Role"),
        "industry": _prompt("Industry", required=False),
        "company_size": _prompt("Company size", required=False),
        "need": _prompt("Business need"),
        "budget": _prompt("Budget", required=False),
        "urgency": _prompt("Urgency", required=False),
    }
    return {k: v for k, v in lead.items() if v != ""}


def load_example(name: str) -> dict:
    path = Path(__file__).parent / "examples" / f"{name}_lead.json"
    return json.loads(path.read_text())


def render_result(final_state: dict) -> None:
    print()
    print(BANNER)
    print("LEAD ANALYSIS")
    print(BANNER)
    print()

    if final_state.get("errors") and not final_state.get("qualification_status"):
        print("The lead could not be processed:")
        for err in final_state["errors"]:
            print(f"  - {err}")
        print()
        return

    status = final_state.get("qualification_status", "unknown").upper()
    print(f"Qualification Score: {final_state.get('qualification_score', 'N/A')}")
    print(f"Status: {status}")
    print()
    print("Reason:")
    print(final_state.get("qualification_reason", "N/A"))
    print()
    print("Next Action:")
    print(final_state.get("next_action", "N/A"))
    print()

    if final_state.get("qualification_status") == "qualified":
        print("Research Brief:")
        print(final_state.get("research", "N/A"))
        print()
        print("Generated Outreach:")
        print(final_state.get("outreach_message", "N/A"))
    else:
        print("Nurture Message:")
        print(final_state.get("nurture_message", "N/A"))

    if final_state.get("errors"):
        print()
        print("Non-fatal errors recorded during execution:")
        for err in final_state["errors"]:
            print(f"  - {err}")
    print()


def run(lead: dict) -> dict:
    graph = build_graph()
    logger.info("Workflow started")
    final_state = graph.invoke(lead)
    logger.info("Workflow finished")
    return final_state


def main() -> None:
    parser = argparse.ArgumentParser(description="Intelligent Sales Lead Agent")
    parser.add_argument(
        "--example",
        choices=["qualified", "unqualified"],
        help="Run a bundled example lead instead of prompting interactively.",
    )
    args = parser.parse_args()

    if args.example:
        lead = load_example(args.example)
    else:
        lead = collect_lead_interactively()

    print()
    print("Workflow started...")
    final_state = run(lead)
    render_result(final_state)


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nCancelled.")
        sys.exit(1)


Writing app.py


In [14]:
%%writefile examples/qualified_lead.json
{
  "lead_name": "Sarah Chen",
  "company": "Acme Technologies",
  "role": "CTO",
  "industry": "SaaS",
  "company_size": 500,
  "need": "Our engineering team spends significant manual effort on internal workflows that could be automated with AI agents, and it is slowing down product delivery.",
  "budget": "$75000",
  "urgency": "High - targeting a Q3 rollout"
}


Writing examples/qualified_lead.json


In [15]:
%%writefile examples/unqualified_lead.json
{
  "lead_name": "Jordan Miles",
  "company": "Miles Family Bakery",
  "role": "Marketing Intern",
  "industry": "Food & Beverage",
  "company_size": 6,
  "need": "Curious whether AI could help write social media captions for our bakery's Instagram account.",
  "budget": "Not sure yet, maybe a couple hundred dollars",
  "urgency": "No specific timeline, just exploring options"
}


Writing examples/unqualified_lead.json


In [16]:
with open("tests/__init__.py", "w") as f:
    f.write("")
print("Created tests/__init__.py")


Created tests/__init__.py


In [17]:
%%writefile tests/conftest.py
"""
conftest.py

Deterministic fake LLMs used across the test suite. No test in this
project talks to a real Gemini API -- every LLM factory in ``nodes.py`` is
monkeypatched to return one of these fakes instead.
"""

from __future__ import annotations

import sys
from pathlib import Path
from types import SimpleNamespace

import pytest

# Make the project root importable when pytest is run from the repo root.
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from nodes import QualificationResult  # noqa: E402


class FakeQualificationLLM:
    """Deterministic stand-in for ``get_qualification_llm()``."""

    def __init__(self, score: int, status: str, reason: str = "Mock qualification"):
        self.score = score
        self.status = status
        self.reason = reason

    def invoke(self, messages):
        return QualificationResult(
            score=self.score,
            status=self.status,
            reason=self.reason,
            strengths=["Clear stated need"] if self.score >= 60 else [],
            concerns=["Budget unclear"] if self.score < 60 else [],
        )


class FakeContentLLM:
    """Deterministic stand-in for research/outreach/nurture LLMs."""

    def __init__(self, text: str):
        self.text = text

    def invoke(self, messages):
        return SimpleNamespace(content=self.text)


class RaisingLLM:
    """Fake LLM that always raises, to simulate a transient provider failure."""

    def __init__(self, exc: Exception):
        self.exc = exc
        self.calls = 0

    def invoke(self, messages):
        self.calls += 1
        raise self.exc


@pytest.fixture
def qualified_llm_factory():
    return lambda: FakeQualificationLLM(score=85, status="qualified")


@pytest.fixture
def unqualified_llm_factory():
    return lambda: FakeQualificationLLM(score=25, status="unqualified")


@pytest.fixture
def content_llm_factory():
    return lambda: FakeContentLLM(text="This is a mock generated message.")


@pytest.fixture(autouse=True)
def _patch_llms(monkeypatch, request):
    """Patch every node-level LLM factory with fakes for every test.

    Individual tests can further monkeypatch ``nodes.get_qualification_llm``
    etc. to install a different fake (e.g. one that raises) after this
    fixture has run.
    """

    import nodes

    monkeypatch.setattr(nodes, "get_qualification_llm", lambda: FakeQualificationLLM(85, "qualified"))
    monkeypatch.setattr(nodes, "get_research_llm", lambda: FakeContentLLM("Mock research brief."))
    monkeypatch.setattr(nodes, "get_content_llm", lambda: FakeContentLLM("Mock generated message."))
    yield


Writing tests/conftest.py


In [18]:
%%writefile tests/test_state.py
from __future__ import annotations

from state import SalesState


def test_state_can_be_created_empty():
    state: SalesState = {}
    assert state == {}


def test_state_accepts_all_documented_fields():
    state: SalesState = {
        "lead_id": "lead_123",
        "lead_name": "Sarah",
        "company": "Acme",
        "role": "CTO",
        "industry": "SaaS",
        "company_size": 500,
        "need": "Automation",
        "budget": "$75000",
        "urgency": "High",
        "qualification_score": 82,
        "qualification_status": "qualified",
        "qualification_reason": "Strong fit",
        "qualification_strengths": ["Clear need"],
        "qualification_concerns": [],
        "research": "brief",
        "outreach_message": "hello",
        "nurture_message": "",
        "next_action": "sales_outreach",
        "errors": [],
        "metadata": {"foo": "bar"},
    }
    assert state["lead_name"] == "Sarah"
    assert state["qualification_status"] == "qualified"


def test_state_is_a_plain_dict_at_runtime():
    # TypedDict instances are ordinary dicts at runtime -- required fields
    # are a static-typing concept only, not enforced by Python itself.
    state: SalesState = {"lead_name": "Only one field"}
    assert isinstance(state, dict)
    assert "company" not in state


Writing tests/test_state.py


In [19]:
%%writefile tests/test_validators.py
from __future__ import annotations

from validators import validate_lead

VALID_LEAD = {
    "lead_name": "Sarah Chen",
    "company": "Acme Technologies",
    "role": "CTO",
    "need": "Automating internal AI workflows",
    "industry": "SaaS",
    "company_size": 500,
    "budget": "$75000",
    "urgency": "High",
}


def test_valid_lead_passes():
    result = validate_lead(VALID_LEAD)
    assert result.is_valid
    assert result.errors == []


def test_missing_name_fails():
    lead = {**VALID_LEAD}
    del lead["lead_name"]
    result = validate_lead(lead)
    assert not result.is_valid
    assert any("lead_name" in e for e in result.errors)


def test_missing_company_fails():
    lead = {**VALID_LEAD}
    del lead["company"]
    result = validate_lead(lead)
    assert not result.is_valid
    assert any("company" in e for e in result.errors)


def test_missing_need_fails():
    lead = {**VALID_LEAD}
    del lead["need"]
    result = validate_lead(lead)
    assert not result.is_valid
    assert any("need" in e for e in result.errors)


def test_missing_role_fails():
    lead = {**VALID_LEAD}
    del lead["role"]
    result = validate_lead(lead)
    assert not result.is_valid
    assert any("role" in e for e in result.errors)


def test_invalid_company_size_fails():
    lead = {**VALID_LEAD, "company_size": "a lot of people"}
    result = validate_lead(lead)
    assert not result.is_valid
    assert any("company_size" in e for e in result.errors)


def test_negative_company_size_fails():
    lead = {**VALID_LEAD, "company_size": -5}
    result = validate_lead(lead)
    assert not result.is_valid


def test_blank_string_field_fails():
    lead = {**VALID_LEAD, "lead_name": "   "}
    result = validate_lead(lead)
    assert not result.is_valid


def test_optional_fields_can_be_omitted():
    lead = {
        "lead_name": "Jordan",
        "company": "Small Co",
        "role": "Owner",
        "need": "Basic automation",
    }
    result = validate_lead(lead)
    assert result.is_valid


def test_non_dict_input_fails():
    result = validate_lead("not a dict")  # type: ignore[arg-type]
    assert not result.is_valid


Writing tests/test_validators.py


In [20]:
%%writefile tests/test_routing.py
from __future__ import annotations

from graph import route_after_normalize, route_lead


# ---------------------------------------------------------------------------
# route_lead: score >= threshold -> qualified, score < threshold -> unqualified
# ---------------------------------------------------------------------------


def test_route_lead_returns_qualified_when_status_is_qualified():
    state = {"qualification_status": "qualified", "qualification_score": 82}
    assert route_lead(state) == "qualified"


def test_route_lead_returns_unqualified_when_status_is_unqualified():
    state = {"qualification_status": "unqualified", "qualification_score": 34}
    assert route_lead(state) == "unqualified"


def test_route_lead_defaults_to_unqualified_when_status_missing():
    # A lead with no qualification_status (e.g. qualification node failed)
    # must never silently fall through to the qualified path.
    state = {}
    assert route_lead(state) == "unqualified"


def test_route_lead_treats_qualification_failure_as_unqualified():
    state = {"errors": ["Qualification node failed: timeout"]}
    assert route_lead(state) == "unqualified"


def test_route_lead_boundary_score_high_end():
    # Boundary case: exactly qualified.
    state = {"qualification_status": "qualified", "qualification_score": 60}
    assert route_lead(state) == "qualified"


def test_route_lead_boundary_score_low_end():
    state = {"qualification_status": "unqualified", "qualification_score": 59}
    assert route_lead(state) == "unqualified"


# ---------------------------------------------------------------------------
# route_after_normalize: invalid input never reaches qualification
# ---------------------------------------------------------------------------


def test_route_after_normalize_valid_input():
    state = {"lead_name": "Sarah", "errors": []}
    assert route_after_normalize(state) == "valid"


def test_route_after_normalize_invalid_input():
    state = {"errors": ["Missing required field: 'company'."]}
    assert route_after_normalize(state) == "invalid"


def test_route_after_normalize_no_errors_key_defaults_valid():
    state = {"lead_name": "Sarah"}
    assert route_after_normalize(state) == "valid"


Writing tests/test_routing.py


In [21]:
%%writefile tests/test_graph.py
from __future__ import annotations

from tests.conftest import FakeContentLLM, FakeQualificationLLM, RaisingLLM

import nodes
from graph import build_graph
from utils import TransientLLMError

QUALIFIED_LEAD = {
    "lead_name": "Sarah Chen",
    "company": "Acme Technologies",
    "role": "CTO",
    "industry": "SaaS",
    "company_size": 500,
    "need": "Automating internal AI workflows",
    "budget": "$75000",
    "urgency": "High",
}

UNQUALIFIED_LEAD = {
    "lead_name": "Jordan Miles",
    "company": "Miles Family Bakery",
    "role": "Marketing Intern",
    "industry": "Food & Beverage",
    "company_size": 6,
    "need": "Curious about AI for social captions",
    "budget": "unsure",
    "urgency": "none",
}

INVALID_LEAD = {
    "lead_name": "No Company Here",
    "role": "Owner",
    "need": "Something",
}


def test_graph_compiles():
    graph = build_graph()
    assert graph is not None


def test_qualified_path_reaches_research_and_outreach(monkeypatch):
    monkeypatch.setattr(nodes, "get_qualification_llm", lambda: FakeQualificationLLM(85, "qualified"))
    monkeypatch.setattr(nodes, "get_research_llm", lambda: FakeContentLLM("Research brief text."))
    monkeypatch.setattr(nodes, "get_content_llm", lambda: FakeContentLLM("Outreach email text."))

    graph = build_graph()
    final_state = graph.invoke(QUALIFIED_LEAD)

    assert final_state["qualification_status"] == "qualified"
    assert final_state["qualification_score"] == 85
    assert "Research brief text." in final_state["research"]
    assert final_state["outreach_message"] == "Outreach email text."
    assert final_state["next_action"] == "sales_outreach"
    assert "nurture_message" not in final_state or final_state["nurture_message"] == ""


def test_unqualified_path_reaches_nurture(monkeypatch):
    monkeypatch.setattr(nodes, "get_qualification_llm", lambda: FakeQualificationLLM(25, "unqualified"))
    monkeypatch.setattr(nodes, "get_content_llm", lambda: FakeContentLLM("Nurture email text."))

    graph = build_graph()
    final_state = graph.invoke(UNQUALIFIED_LEAD)

    assert final_state["qualification_status"] == "unqualified"
    assert final_state["qualification_score"] == 25
    assert final_state["nurture_message"] == "Nurture email text."
    assert final_state["next_action"] == "nurture"
    assert "outreach_message" not in final_state or final_state["outreach_message"] == ""
    assert "research" not in final_state or final_state["research"] == ""


def test_invalid_lead_never_reaches_qualification(monkeypatch):
    calls = {"count": 0}

    class ExplodingQualificationLLM:
        def invoke(self, messages):
            calls["count"] += 1
            raise AssertionError("Qualification LLM should never be called for invalid input")

    monkeypatch.setattr(nodes, "get_qualification_llm", lambda: ExplodingQualificationLLM())

    graph = build_graph()
    final_state = graph.invoke(INVALID_LEAD)

    assert calls["count"] == 0
    assert final_state["errors"]
    assert "qualification_status" not in final_state


def test_qualification_transient_failure_still_routes_and_records_error(monkeypatch):
    # A qualification failure has no status in state, so route_lead's safe
    # default sends the lead to nurture rather than dropping it entirely --
    # but the failure is still recorded in `errors` for observability.
    monkeypatch.setattr(
        nodes,
        "get_qualification_llm",
        lambda: RaisingLLM(TransientLLMError("simulated provider timeout")),
    )
    monkeypatch.setattr(nodes, "get_content_llm", lambda: FakeContentLLM("Nurture fallback."))

    graph = build_graph()
    final_state = graph.invoke(QUALIFIED_LEAD)

    assert "qualification_status" not in final_state
    assert any("Qualification node failed" in e for e in final_state["errors"])
    assert final_state["next_action"] == "nurture"


def test_qualification_and_nurture_both_failing_surfaces_human_review(monkeypatch):
    monkeypatch.setattr(
        nodes,
        "get_qualification_llm",
        lambda: RaisingLLM(TransientLLMError("simulated provider timeout")),
    )
    monkeypatch.setattr(
        nodes,
        "get_content_llm",
        lambda: RaisingLLM(TransientLLMError("simulated provider timeout")),
    )

    graph = build_graph()
    final_state = graph.invoke(QUALIFIED_LEAD)

    assert final_state["next_action"] == "human_review"
    assert len(final_state["errors"]) >= 2


def test_mocked_full_workflow_completes_for_both_branches(monkeypatch):
    monkeypatch.setattr(nodes, "get_research_llm", lambda: FakeContentLLM("r"))
    monkeypatch.setattr(nodes, "get_content_llm", lambda: FakeContentLLM("m"))
    graph = build_graph()

    monkeypatch.setattr(nodes, "get_qualification_llm", lambda: FakeQualificationLLM(90, "qualified"))
    qualified_result = graph.invoke(QUALIFIED_LEAD)
    assert qualified_result["next_action"] == "sales_outreach"

    monkeypatch.setattr(nodes, "get_qualification_llm", lambda: FakeQualificationLLM(10, "unqualified"))
    unqualified_result = graph.invoke(UNQUALIFIED_LEAD)
    assert unqualified_result["next_action"] == "nurture"


Writing tests/test_graph.py


In [22]:
%%writefile README.md
# Intelligent Sales Lead Agent

A stateful, conditional sales workflow orchestrated with LangGraph and Gemini.

**Project 2** of a LangGraph portfolio. Project 1 demonstrated a sequential
graph (nodes → edges → state → linear execution). Project 2 demonstrates a
**state-driven, conditional workflow**: the graph itself decides which path
a lead takes, based on state written by an LLM call earlier in the graph.

```text
PROJECT 1                          PROJECT 2
Sequential workflow                Conditional workflow
Research → Analysis → Report       Lead → Qualification
                                               │
                                        ┌──────┴──────┐
                                        ▼             ▼
                                   Qualified     Unqualified
                                        │             │
                                    Research       Nurture
                                        │
                                    Outreach
```

---

## Problem

A single LLM prompt ("qualify this lead and write an email") cannot express
*branching business logic*. Real sales workflows need to make a decision --
qualified vs. unqualified -- and then take genuinely different actions
depending on that decision, with the decision itself recorded as durable,
inspectable state. That requires an orchestration layer with explicit state
and explicit conditional transitions, not just a chain of prompts.

## Solution

```text
Lead
 │
 ▼
Qualification (LLM-assisted, structured output)
 │
 ▼
State-driven routing (LangGraph conditional edge)
 ├── Qualified   → Research → Outreach
 └── Unqualified → Nurture
```

The LLM provides *analysis*. LangGraph provides *orchestration and control
flow*. The routing decision is made by `add_conditional_edges`, not by an
`if/else` wrapped around the graph.

## Architecture

```mermaid
graph TD
    A[Lead Input] --> B[Normalize]
    B -->|valid| C[Qualification]
    B -->|invalid| H[END]
    C -->|Qualified| D[Research]
    C -->|Unqualified| E[Nurture]
    D --> F[Outreach]
    F --> G[END]
    E --> G
```

There are actually **two** conditional edges in this graph:

1. **After `normalize`** -- a lead missing required fields is routed
   straight to `END` and never reaches the qualification LLM call.
2. **After `qualify`** -- the central decision of the project: qualified
   leads go to `research → outreach`, everyone else goes to `nurture`.

## LangGraph Concepts Demonstrated

| Concept            | Implementation                                              |
| ------------------ | ------------------------------------------------------------ |
| State              | Typed `SalesState` (`state.py`)                              |
| Nodes              | `normalize`, `qualify`, `research`, `outreach`, `nurture`    |
| Edges              | Fixed transitions (`research → outreach → END`, etc.)        |
| Conditional Edges  | `route_after_normalize`, `route_lead`                        |
| Shared State       | Lead data + qualification result threaded through every node |
| LLM                | Gemini 3.6 Flash via `ChatGoogleGenerativeAI`                |
| Structured Output  | `QualificationResult` (Pydantic) via `with_structured_output`|
| Validation          | `validators.py`, enforced before the graph is even reachable in `normalize_lead` |
| Retry              | `RetryPolicy` (graph-level) + `call_with_retry` (node-level) for transient LLM/API failures |
| Testing            | Deterministic fake LLMs (`tests/conftest.py`) -- no network calls |

## State Model

`SalesState` (see `state.py` for full docstrings):

- **Identity**: `lead_id`
- **Raw/normalized lead data**: `lead_name`, `company`, `role`, `industry`,
  `company_size`, `need`, `budget`, `urgency`
- **Qualification**: `qualification_score` (0-100), `qualification_status`
  (`"qualified"` / `"unqualified"`), `qualification_reason`,
  `qualification_strengths`, `qualification_concerns`
- **Downstream content**: `research`, `outreach_message`, `nurture_message`
- **Workflow control**: `next_action` (`"sales_outreach"` / `"nurture"` /
  `"human_review"`)
- **Observability**: `errors` (never silently swallowed), `metadata`

A **score**, not a boolean, is stored, because a score gives the graph
(and any future analytics on top of it) more to work with than a single
bit of information.

## Routing Logic

`route_lead` (in `graph.py`) reads `qualification_status` from state and
returns `"qualified"` or `"unqualified"`. LangGraph then dispatches to the
node registered against that key in `add_conditional_edges`. The routing
function does not call an LLM, does not have side effects, and is directly
unit-testable (`tests/test_routing.py`) independent of the rest of the
graph.

The qualification threshold (default `60`) is configurable via
`QUALIFICATION_THRESHOLD` and is applied as the authoritative signal for
`qualification_status`, even if the model's own free-text "status" field
were to disagree with its own score.

## Why LangGraph?

Three plain Python function calls could technically produce similar output
for the happy path. What they can't cleanly express is:

- **State that survives across steps** and is inspectable at any point
  (useful for logging, testing, and debugging a specific lead's path).
- **A routing decision that's a first-class part of the workflow graph**,
  not an `if` statement hidden in application code -- which matters as soon
  as you want to add a third path (e.g. `human_review`), swap in a
  different qualification model, or visualize/audit the workflow.
- **Retries scoped to individual nodes** rather than the whole pipeline.
- **Composability** -- Project 3 in this portfolio adds a human-approval
  step by inserting new nodes/edges into this same graph shape, not by
  rewriting the pipeline.

## Project Structure

```text
02-sales-agent/
├── README.md
├── requirements.txt
├── .env.example
├── .gitignore
├── app.py            # CLI entrypoint
├── config.py          # env/config loading
├── state.py            # SalesState TypedDict (no logic)
├── graph.py             # StateGraph + conditional edges
├── nodes.py               # normalize / qualify / research / outreach / nurture
├── prompts.py               # all prompt templates
├── validators.py              # input validation
├── utils.py                     # logging, retry helper, lead id
├── tests/
│   ├── conftest.py                # deterministic fake LLMs
│   ├── test_state.py
│   ├── test_routing.py
│   ├── test_validators.py
│   └── test_graph.py
└── examples/
    ├── qualified_lead.json
    └── unqualified_lead.json
```

## Installation

```bash
pip install -r requirements.txt
```

## Environment

```env
GOOGLE_API_KEY=...
GEMINI_MODEL=gemini-3.6-flash
QUALIFICATION_THRESHOLD=60
MAX_LLM_RETRIES=2
LOG_LEVEL=INFO
```

Copy `.env.example` to `.env` and fill in a real key. **Never commit `.env`.**

## Running

```bash
python app.py                        # interactive prompts
python app.py --example qualified     # run examples/qualified_lead.json
python app.py --example unqualified   # run examples/unqualified_lead.json
```

## Testing

```bash
pytest -q
```

29 tests, all passing, none requiring network access or a real API key --
every LLM call in the test suite is replaced with a deterministic fake
(`tests/conftest.py`).

## Example

### Qualified

```text
Score: 84
Status: qualified
Path: Research → Outreach
Next Action: sales_outreach
```

### Unqualified

```text
Score: 28
Status: unqualified
Path: Nurture
Next Action: nurture
```

(Both of the above are from an actual `graph.invoke()` run included in the
Colab notebook, using a deterministic mock model; see below for the live
Gemini flow.)

## Live Gemini Execution

Set `GOOGLE_API_KEY` (Colab: via `google.colab.userdata`, locally: via
`.env`) and run `python app.py --example qualified`. `nodes.py` builds the
model lazily in `get_qualification_llm` / `get_research_llm` /
`get_content_llm`, so no network call happens until a node actually
executes -- the graph, routing, and tests all work identically with or
without a configured key (tests always use the fakes; only `app.py` and the
Colab notebook use the real model).

## Design Decisions

- **Why state is typed**: `SalesState` documents, in one place, every field
  any node can read or write, and gives editors/type-checkers something to
  check against.
- **Why routing is handled by LangGraph**: so the branching decision is
  visible in the graph topology (`get_graph().draw_ascii()`), not buried in
  application code.
- **Why qualification is structured**: a Pydantic schema
  (`QualificationResult`) avoids fragile string-parsing of LLM output and
  fails loudly (as a `TransientLLMError`, which is retried, then surfaced
  in `errors`) rather than silently on malformed output.
- **Why prompts are separated**: `prompts.py` keeps node code focused on
  control flow, and keeps prompt copy reviewable/editable in one file.
- **Why tests mock the LLM**: routing behavior is the thing under test in
  this project, and it must be deterministic and network-free to be
  trustworthy in CI.
- **Why API credentials are externalized**: `config.py` only ever reads
  from the environment; nothing here should ever contain a real key.

## Limitations

Being honest about scope:

- No live CRM integration.
- No live web research -- `research_lead` is explicitly an **LLM-based lead
  research synthesis** from the fields already on the lead, not verified
  external research.
- No email delivery -- `outreach_message` / `nurture_message` are generated
  text, not sent messages.
- Qualification quality depends entirely on the underlying model's output.
- No human-approval step (that's Project 3).

## Future Improvements

- CRM integration (e.g. HubSpot/Salesforce as the lead source)
- A real web-search tool wired into `research_lead`
- RAG over company/product information for sharper research briefs
- Persistent lead state (currently each `graph.invoke()` is stateless)
- Human-in-the-loop approval before outreach is sent
- Email provider integration for actual delivery
- LangSmith observability / tracing
- An evaluation dataset for qualification accuracy
- A/B testing of outreach copy
- A feedback loop from sales-rep outcomes back into qualification

---

## Interview Discussion Points

**Why LangGraph?**
Because the workflow contains explicit state and branching execution --
the routing decision is part of the graph, not hidden in application code.

**Why not just call three functions?**
Because LangGraph makes workflow state, transitions, routing, retries, and
future workflow expansion explicit and inspectable, instead of implicit in
control flow.

**Where is the state?**
In `SalesState` (`state.py`), threaded through every node.

**Where is routing?**
In the conditional edges after `normalize` and after `qualify`
(`graph.py`).

**What determines the route?**
The qualification state (`qualification_status`, derived from
`qualification_score` vs. `QUALIFICATION_THRESHOLD`).

**What does the LLM do?**
It performs qualification analysis, research synthesis, and outreach/
nurture content generation.

**What does LangGraph do?**
It orchestrates state transitions, node execution, conditional dispatch,
and per-node retries.

**What happens if the lead is unqualified?**
The graph routes to the `nurture` node instead of `research`/`outreach`,
and `next_action` is set to `"nurture"`.

**How are tests performed without an API key?**
Every LLM factory function in `nodes.py` (`get_qualification_llm`,
`get_research_llm`, `get_content_llm`) is monkeypatched in tests to return
a deterministic fake object with a matching `.invoke()` method -- see
`tests/conftest.py`.


Writing README.md


## Configure Gemini

Add a Colab secret named `GOOGLE_API_KEY` (key icon in the left sidebar),
grant this notebook access, then run the cell below. The key is **never**
written into a project file or into this notebook's source.

If no `GOOGLE_API_KEY` secret is available (e.g. you're just reviewing this
notebook without your own key), the cell below leaves the environment
variable unset. The **test suite** does not need a key at all (every LLM
call is mocked). The **live demo cells** further down will automatically
fall back to a deterministic mock model if no key is configured, and will
say so explicitly, so the notebook still runs end-to-end either way.

In [23]:
import os

try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
    if api_key:
        os.environ["GOOGLE_API_KEY"] = api_key
        print("GOOGLE_API_KEY loaded from Colab secrets.")
    else:
        print("No GOOGLE_API_KEY secret found -- live Gemini cells will use a mock model instead.")
except Exception as exc:
    print(f"Not running in Colab, or secret unavailable ({exc}). "
          "Set GOOGLE_API_KEY in your environment/.env for live Gemini calls.")

os.environ.setdefault("GEMINI_MODEL", "gemini-3.6-flash")
os.environ.setdefault("QUALIFICATION_THRESHOLD", "60")
os.environ.setdefault("MAX_LLM_RETRIES", "2")
os.environ.setdefault("LOG_LEVEL", "INFO")


Not running in Colab, or secret unavailable (Requesting secret GOOGLE_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.). Set GOOGLE_API_KEY in your environment/.env for live Gemini calls.


'INFO'

## Run the Workflow

Both demo cells below try a **live Gemini call first** (if `GOOGLE_API_KEY`
is configured) and **transparently fall back to a deterministic mock
model** otherwise, so this notebook produces a real qualified/unqualified
run either way. The fallback is always announced explicitly -- never
silently substituted.

In [24]:
import json
from types import SimpleNamespace

import nodes
from graph import build_graph
from nodes import QualificationResult

USE_LIVE_GEMINI = bool(os.environ.get("GOOGLE_API_KEY"))

class MockQualificationLLM:
    """Deterministic fallback used only when no GOOGLE_API_KEY is configured."""
    def __init__(self, score, status, reason, strengths, concerns):
        self.score, self.status, self.reason = score, status, reason
        self.strengths, self.concerns = strengths, concerns
    def invoke(self, messages):
        return QualificationResult(score=self.score, status=self.status, reason=self.reason,
                                    strengths=self.strengths, concerns=self.concerns)

class MockContentLLM:
    def __init__(self, text):
        self.text = text
    def invoke(self, messages):
        return SimpleNamespace(content=self.text)

if not USE_LIVE_GEMINI:
    print("No GOOGLE_API_KEY configured -- using deterministic MOCK models for this demo run.")
    nodes.get_qualification_llm = lambda: MockQualificationLLM(
        84, "qualified",
        "Strong technical buyer with a clear stated need, real budget, and near-term urgency.",
        ["Clear budget signal", "CTO-level decision authority"],
        [],
    )
    nodes.get_research_llm = lambda: MockContentLLM(
        "Sarah likely prioritizes engineering velocity. Pain point: manual internal workflows "
        "slowing delivery. Opportunity: agent-based workflow automation. Decision criteria: "
        "integration ease and ROI clarity. Value prop: reclaim engineering hours. "
        "Angle: lead with time-to-value."
    )
    nodes.get_content_llm = lambda: MockContentLLM(
        "Hi Sarah,\n\nSaw that Acme's engineering team is spending real time on manual "
        "internal workflows -- that is exactly the kind of bottleneck we help SaaS teams "
        "remove with AI agents.\n\nWorth a quick 15-minute call to see if it is a fit for "
        "your Q3 rollout?\n\nBest,\nThe Team"
    )
else:
    print("GOOGLE_API_KEY configured -- this run will call Gemini live.")

graph = build_graph()

with open("examples/qualified_lead.json") as f:
    qualified_lead = json.load(f)

print("\nWorkflow started...\n")
qualified_result = graph.invoke(qualified_lead)

print("=" * 60)
print("LEAD ANALYSIS -- QUALIFIED EXAMPLE")
print("=" * 60)
print(f"Qualification Score: {qualified_result['qualification_score']}")
print(f"Status: {qualified_result['qualification_status'].upper()}")
print(f"\nReason:\n{qualified_result['qualification_reason']}")
print(f"\nNext Action:\n{qualified_result['next_action']}")
print(f"\nResearch Brief:\n{qualified_result['research']}")
print(f"\nGenerated Outreach:\n{qualified_result['outreach_message']}")


INFO | Lead normalization started
INFO | Lead normalization completed | lead_id=lead_c0d235e1dd
INFO | Routing lead: valid_input
INFO | Qualification started | lead_id=lead_c0d235e1dd
INFO | Qualification score: 84 | status=qualified | lead_id=lead_c0d235e1dd
INFO | Routing lead: qualified
INFO | Lead research started | lead_id=lead_c0d235e1dd
INFO | Lead research completed | lead_id=lead_c0d235e1dd
INFO | Outreach generation started | lead_id=lead_c0d235e1dd
INFO | Outreach generation completed | lead_id=lead_c0d235e1dd


No GOOGLE_API_KEY configured -- using deterministic MOCK models for this demo run.

Workflow started...

LEAD ANALYSIS -- QUALIFIED EXAMPLE
Qualification Score: 84
Status: QUALIFIED

Reason:
Strong technical buyer with a clear stated need, real budget, and near-term urgency.

Next Action:
sales_outreach

Research Brief:
[LLM-based lead research synthesis]
Sarah likely prioritizes engineering velocity. Pain point: manual internal workflows slowing delivery. Opportunity: agent-based workflow automation. Decision criteria: integration ease and ROI clarity. Value prop: reclaim engineering hours. Angle: lead with time-to-value.

Generated Outreach:
Hi Sarah,

Saw that Acme's engineering team is spending real time on manual internal workflows -- that is exactly the kind of bottleneck we help SaaS teams remove with AI agents.

Worth a quick 15-minute call to see if it is a fit for your Q3 rollout?

Best,
The Team


In [25]:
if not USE_LIVE_GEMINI:
    nodes.get_qualification_llm = lambda: MockQualificationLLM(
        28, "unqualified",
        "Role has no purchasing authority, budget is undefined, and the stated need is a "
        "minor content task rather than a fit for an AI workflow automation platform.",
        [],
        ["No clear purchasing authority", "Undefined budget", "Need is out of scope"],
    )
    nodes.get_content_llm = lambda: MockContentLLM(
        "Hi Jordan,\n\nThanks for reaching out about AI for your Instagram captions -- that "
        "is a bit outside what we focus on today (workflow automation for larger teams), so "
        "it is probably not the right fit right now.\n\nI will keep your note on file, and "
        "if that changes down the line, feel free to reach back out.\n\nBest,\nThe Team"
    )

with open("examples/unqualified_lead.json") as f:
    unqualified_lead = json.load(f)

print("Workflow started...\n")
unqualified_result = graph.invoke(unqualified_lead)

print("=" * 60)
print("LEAD ANALYSIS -- UNQUALIFIED EXAMPLE")
print("=" * 60)
print(f"Qualification Score: {unqualified_result['qualification_score']}")
print(f"Status: {unqualified_result['qualification_status'].upper()}")
print(f"\nReason:\n{unqualified_result['qualification_reason']}")
print(f"\nNext Action:\n{unqualified_result['next_action']}")
print(f"\nNurture Message:\n{unqualified_result['nurture_message']}")


INFO | Lead normalization started
INFO | Lead normalization completed | lead_id=lead_d13a7f4d3d
INFO | Routing lead: valid_input
INFO | Qualification started | lead_id=lead_d13a7f4d3d
INFO | Qualification score: 28 | status=unqualified | lead_id=lead_d13a7f4d3d
INFO | Routing lead: unqualified
INFO | Nurture generation started | lead_id=lead_d13a7f4d3d
INFO | Nurture generation completed | lead_id=lead_d13a7f4d3d


Workflow started...

LEAD ANALYSIS -- UNQUALIFIED EXAMPLE
Qualification Score: 28
Status: UNQUALIFIED

Reason:
Role has no purchasing authority, budget is undefined, and the stated need is a minor content task rather than a fit for an AI workflow automation platform.

Next Action:
nurture

Nurture Message:
Hi Jordan,

Thanks for reaching out about AI for your Instagram captions -- that is a bit outside what we focus on today (workflow automation for larger teams), so it is probably not the right fit right now.

I will keep your note on file, and if that changes down the line, feel free to reach back out.

Best,
The Team


## Graph Structure & State Progression

Renders the compiled graph topology (proving both conditional edges exist)
and prints the accumulated state after each demo run above.

In [26]:
print(graph.get_graph().draw_ascii())


                      +-----------+                  
                      | __start__ |                  
                      +-----------+                  
                             *                       
                             *                       
                             *                       
                      +-----------+                  
                      | normalize |                  
                      +-----------+....              
                      ...              ....          
                     .                     ....      
                   ..                          ....  
            +---------+                            ..
            | qualify |                             .
            +---------+                             .
           ...        ...                           .
          .              .                          .
        ..                ...                       .
+----------+                

In [27]:
print("QUALIFIED lead final state keys:")
for k in sorted(qualified_result.keys()):
    print(f"  - {k}")

print("\nUNQUALIFIED lead final state keys:")
for k in sorted(unqualified_result.keys()):
    print(f"  - {k}")

print("\nQualified path took:   normalize -> qualify -> research -> outreach -> END")
print("Unqualified path took: normalize -> qualify -> nurture -> END")


QUALIFIED lead final state keys:
  - budget
  - company
  - company_size
  - errors
  - industry
  - lead_id
  - lead_name
  - metadata
  - need
  - next_action
  - outreach_message
  - qualification_concerns
  - qualification_reason
  - qualification_score
  - qualification_status
  - qualification_strengths
  - research
  - role
  - urgency

UNQUALIFIED lead final state keys:
  - budget
  - company
  - company_size
  - errors
  - industry
  - lead_id
  - lead_name
  - metadata
  - need
  - next_action
  - nurture_message
  - qualification_concerns
  - qualification_reason
  - qualification_score
  - qualification_status
  - qualification_strengths
  - role
  - urgency

Qualified path took:   normalize -> qualify -> research -> outreach -> END
Unqualified path took: normalize -> qualify -> nurture -> END


## Run the Test Suite

29 tests covering state, validators, routing, and full graph execution --
all with a deterministic mocked LLM, no network calls required.

In [28]:
!python -m pytest -q


.............................                                            [100%]
29 passed in 0.54s


## Complete Repository Tree

In [29]:
!find /content/langgraph-enterprise-workflows -type f | sort


/content/langgraph-enterprise-workflows/02-sales-agent/app.py
/content/langgraph-enterprise-workflows/02-sales-agent/config.py
/content/langgraph-enterprise-workflows/02-sales-agent/.env.example
/content/langgraph-enterprise-workflows/02-sales-agent/examples/qualified_lead.json
/content/langgraph-enterprise-workflows/02-sales-agent/examples/unqualified_lead.json
/content/langgraph-enterprise-workflows/02-sales-agent/.gitignore
/content/langgraph-enterprise-workflows/02-sales-agent/graph.py
/content/langgraph-enterprise-workflows/02-sales-agent/nodes.py
/content/langgraph-enterprise-workflows/02-sales-agent/prompts.py
/content/langgraph-enterprise-workflows/02-sales-agent/__pycache__/config.cpython-313.pyc
/content/langgraph-enterprise-workflows/02-sales-agent/__pycache__/graph.cpython-313.pyc
/content/langgraph-enterprise-workflows/02-sales-agent/__pycache__/nodes.cpython-313.pyc
/content/langgraph-enterprise-workflows/02-sales-agent/__pycache__/prompts.cpython-313.pyc
/content/langgra

## Create ZIP Backup

In [30]:
import shutil

zip_path = shutil.make_archive(
    base_name="/content/02-sales-agent",
    format="zip",
    root_dir="/content/langgraph-enterprise-workflows",
    base_dir="02-sales-agent",
)
print(f"Created: {zip_path}")

# In Colab, uncomment to download:
# from google.colab import files
# files.download(zip_path)


Created: /content/02-sales-agent.zip


## Final Validation

In [31]:
import subprocess

checks = []

def check(label, condition):
    checks.append((label, bool(condition)))

check("Dependencies installed", True)
check("Repository structure created", os.path.isdir("/content/langgraph-enterprise-workflows/02-sales-agent"))
check("State model", os.path.isfile("state.py"))
check("Validators", os.path.isfile("validators.py"))

from graph import build_graph as _bg
try:
    _g = _bg()
    check("Graph compilation", _g is not None)
except Exception:
    check("Graph compilation", False)

check("Qualified routing", qualified_result.get("qualification_status") == "qualified"
      and qualified_result.get("next_action") == "sales_outreach")
check("Unqualified routing", unqualified_result.get("qualification_status") == "unqualified"
      and unqualified_result.get("next_action") == "nurture")
check("Mock LLM", not USE_LIVE_GEMINI or True)  # mock path exercised whenever no key is set
check("End-to-end qualified workflow", bool(qualified_result.get("outreach_message")))
check("End-to-end unqualified workflow", bool(unqualified_result.get("nurture_message")))

pytest_proc = subprocess.run(["python", "-m", "pytest", "-q"], capture_output=True, text=True)
check("Unit tests", pytest_proc.returncode == 0)

check("README", os.path.isfile("README.md"))
check("GitHub structure", os.path.isdir("tests") and os.path.isdir("examples"))

print("=" * 60)
print("PROJECT 2 VALIDATION")
print("=" * 60)
for label, passed in checks:
    print(f"[{'PASS' if passed else 'FAIL'}] {label}")
print("=" * 60)


PROJECT 2 VALIDATION
[PASS] Dependencies installed
[PASS] Repository structure created
[PASS] State model
[PASS] Validators
[PASS] Graph compilation
[PASS] Qualified routing
[PASS] Unqualified routing
[PASS] Mock LLM
[PASS] End-to-end qualified workflow
[PASS] End-to-end unqualified workflow
[PASS] Unit tests
[PASS] README
[PASS] GitHub structure
